# C-MAPSS model experiment runner

In [5]:
import os
import sys
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path

In [6]:
current_directory = Path.cwd().resolve()
PROJECT_ROOT = next((directory for directory in (current_directory, *current_directory.parents)if (directory / 'pyproject.toml').is_file()), None)

if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root')

CODE_ROOT = PROJECT_ROOT / 'code'

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

os.chdir(PROJECT_ROOT)
os.environ.setdefault('MLFLOW_ALLOW_FILE_STORE', 'true')
os.environ.setdefault('MLFLOW_TRACKING_URI', f'file:{PROJECT_ROOT / "mlruns"}')
os.environ.setdefault('CMAPSS_MLFLOW_EXPERIMENT', 'cmapss-preprocessing')
os.environ.setdefault('CMAPSS_MLFLOW_TRAINING_EXPERIMENT', 'cmapss-training')

print(f'Project root: {PROJECT_ROOT}')

Project root: /home/aanchal/nasa_c_mapss


In [7]:
from src.training.lstm import train_lstm
from src.build_spark import spark_session_context
from src.training.tree_models import train_random_forest
from src.data_processing.preprocessing import run_subset_preprocessing
from src.training.tree_models.run_tabular_training import run_training
from src.training.evaluation.test_evaluation import evaluate_test_data

from src.data_processing import build_endpoint_sequences
from src.tracking import load_training_feature_columns, load_training_model
from src.training.evaluation.engine_endpoint_evaluation import (evaluate_prediction_diagnostics,
                                                                prepare_pseudo_test_validation,
                                                                select_pseudo_test_endpoints)
from src.training.evaluation.model_evaluation_metrics import evaluate_predictions

In [8]:
SUBSETS = ('FD001', 'FD002', 'FD003', 'FD004')
RAW_DATA_DIR = PROJECT_ROOT / 'Data' / 'CMAPSSData'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'Data' / 'processed'
RUL_CAP = 125

BASELINE_PREPROCESSING_RUN_IDS = {
    'FD001': '208b1c6c99264e74a42f35df2edfd4dd',
    'FD002': '1030d9d2baf7491e901e85ed69d5655f',
    'FD003': '61dd784103e24f408aea4486cb982a35',
    'FD004': '432ff4df92c7451b97e52832049b5a70',
}
TEMPORAL_PREPROCESSING_RUN_IDS = {
    'FD001': '20db40e34e4b42f392e931eab4ae1b7d',
    'FD002': '324277ebf695472886eaf129724e9833',
    'FD003': '246b7b1ac64d455298f08db0b0abd11d',
    'FD004': '0bd8c9d6df084181b95c5048faa9ad5f',
}

RUN_TESTS = True
RUN_TEMPORAL_PREPROCESSING = False
RUN_TRAINING = True

In [ ]:
preprocessing_run_ids = {(subset, 'baseline'): run_id for subset, run_id in BASELINE_PREPROCESSING_RUN_IDS.items()}
preprocessing_run_ids.update({(subset, 'temporal'): run_id for subset, run_id in TEMPORAL_PREPROCESSING_RUN_IDS.items()})

preprocessing_results = [{'subset': subset, 
                          'feature_set': 'baseline', 
                          'preprocessing_run_id': run_id, 
                          'source': 'existing'}
                         for subset, run_id in BASELINE_PREPROCESSING_RUN_IDS.items()]

preprocessing_results.extend({'subset': subset,
                              'feature_set': 'temporal',
                              'preprocessing_run_id': run_id,
                              'source': 'existing'}
                             for subset, run_id in TEMPORAL_PREPROCESSING_RUN_IDS.items())

if RUN_TEMPORAL_PREPROCESSING:
    for subset in SUBSETS:
        print(f'\n[{subset}] preprocessing temporal features...', flush=True)
        
        with spark_session_context(app_name=f'cmapss-{subset}-temporal-notebook') as spark:
            
            result = run_subset_preprocessing(spark=spark, 
            subset=subset, 
            raw_data_dir=RAW_DATA_DIR, 
            output_dir=PROCESSED_DATA_DIR, 
            include_temporal_features=True)
        
        preprocessing_run_ids[(subset, 'temporal')] = result.run_id
        
        preprocessing_results.append({'subset': subset,
                                      'feature_set': 'temporal',
                                      'preprocessing_run_id': result.run_id,
                                      'source': 'created',
                                      'feature_count': result.feature_count,
                                      'train_rows': result.train_row_count})

pd.DataFrame(preprocessing_results)

In [ ]:
training_results = []

if RUN_TRAINING:
    required_preprocessing_runs = {(subset, feature_set) for subset in SUBSETS for feature_set in ('baseline', 'temporal')}
    
    missing_runs = required_preprocessing_runs - preprocessing_run_ids.keys()
    if missing_runs:
        raise RuntimeError(f'Missing preprocessing runs: {sorted(missing_runs)}')

    for feature_set in ('baseline', 'temporal'):
        for subset in SUBSETS:
            preprocessing_run_id = preprocessing_run_ids[(subset, feature_set)]

            print(f'\n[{subset}] training RF with {feature_set} features...', flush=True)
            
            rf_run_id = train_random_forest(subset_id=subset, preprocessing_run_id=preprocessing_run_id, 
                                            processed_data_dir=PROCESSED_DATA_DIR, rul_cap=RUL_CAP)
            
            rf_test_metrics = evaluate_test_data(subset_id=subset, training_run_id=rf_run_id, 
                                                 processed_data_dir=PROCESSED_DATA_DIR, raw_data_dir=RAW_DATA_DIR)
            
            training_results.append({
                    'subset': subset,
                    'model': 'random_forest',
                    'feature_set': feature_set,
                    'preprocessing_run_id': preprocessing_run_id,
                    'training_run_id': rf_run_id,
                    **{f'test_{name}': value for name, value in rf_test_metrics.items()}})

            print(f'\n[{subset}] training XGBoost with {feature_set} features...', flush=True)
            
            xgboost_result = run_training( model_type='xgboost', subset_id=subset, preprocessing_run_id=preprocessing_run_id, 
                                          processed_data_dir=PROCESSED_DATA_DIR, raw_data_dir=RAW_DATA_DIR)
            
            training_results.append({
                    'subset': subset,
                    'model': 'xgboost',
                    'feature_set': feature_set,
                    'preprocessing_run_id': preprocessing_run_id,
                    'training_run_id': xgboost_result['training_run_id'],
                    **{f'test_{name}': value for name, value in xgboost_result['test_metrics'].items()}})

    for subset in SUBSETS:        
        preprocessing_run_id = preprocessing_run_ids[(subset, 'baseline')]
        
        print(f'\n[{subset}] training capped-target LSTM...', flush=True)
        
        lstm_result = train_lstm(subset_id=subset, preprocessing_run_id=preprocessing_run_id, 
                                 processed_data_dir=PROCESSED_DATA_DIR, raw_data_dir=RAW_DATA_DIR, rul_cap=RUL_CAP)
        
        training_results.append({
                'subset': subset,
                'model': 'lstm',
                'feature_set': 'base_sequence',
                'preprocessing_run_id': preprocessing_run_id,
                'training_run_id': lstm_result['training_run_id'],
                **{f'test_{name}': value for name, value in lstm_result['test_metrics'].items()}})

results = pd.DataFrame(training_results)

results

In [ ]:
comparison_columns = [
    'test_mae',
    'test_rmse',
    'test_nasa_score',
    'test_bias',
    'test_late_prediction_rate',
    'test_worst_positive_error',
    'test_worst_negative_error',
    'test_largest_nasa_contribution',
    'test_top_3_nasa_contribution',
]

results.set_index(['subset', 'model', 'feature_set'])[comparison_columns].round(3)

In [ ]:
temporal_lstm_results = []

for subset in SUBSETS:
    preprocessing_run_id = TEMPORAL_PREPROCESSING_RUN_IDS[subset]
    print(f'\n[{subset}] training capped-target LSTM with temporal features...', flush=True)
    
    result = train_lstm(subset_id=subset, preprocessing_run_id=preprocessing_run_id, 
                        processed_data_dir=PROCESSED_DATA_DIR, raw_data_dir=RAW_DATA_DIR, 
                        rul_cap=RUL_CAP, feature_set='temporal_sequence')
    
    temporal_lstm_results.append({
            'subset': subset,
            'model': 'lstm',
            'feature_set': 'temporal_sequence',
            'preprocessing_run_id': preprocessing_run_id,
            'training_run_id': result['training_run_id'],
            **{f'test_{name}': value for name, value in result['test_metrics'].items()}})

pd.DataFrame(temporal_lstm_results)

In [9]:
ROBUSTNESS_SEEDS = tuple(range(42, 52))

ROBUSTNESS_CANDIDATES = (
    {'subset': 'FD001', 'candidate': 'temporal_lstm', 'role': 'champion', 'model_family': 'lstm', 'preprocessing_run_id': TEMPORAL_PREPROCESSING_RUN_IDS['FD001'], 'training_run_id': '9d42f953ae0540a2b65d053137fd6361'},
    {'subset': 'FD001', 'candidate': 'temporal_xgboost', 'role': 'competitor', 'model_family': 'tabular', 'preprocessing_run_id': TEMPORAL_PREPROCESSING_RUN_IDS['FD001'], 'training_run_id': 'd37624ea98f5459db673a69c5dc56951'},
    {'subset': 'FD002', 'candidate': 'temporal_lstm', 'role': 'champion', 'model_family': 'lstm', 'preprocessing_run_id': TEMPORAL_PREPROCESSING_RUN_IDS['FD002'], 'training_run_id': '490407b61beb40ffa97c0572bd04fc0c'},
    {'subset': 'FD002', 'candidate': 'temporal_xgboost', 'role': 'competitor', 'model_family': 'tabular', 'preprocessing_run_id': TEMPORAL_PREPROCESSING_RUN_IDS['FD002'], 'training_run_id': '1b7584f00aad4746a59ff0103f2448c3'},
    {'subset': 'FD003', 'candidate': 'base_lstm', 'role': 'champion', 'model_family': 'lstm', 'preprocessing_run_id': BASELINE_PREPROCESSING_RUN_IDS['FD003'], 'training_run_id': '8f7a7a1c48ef4e11874b8555b6c7f4c7'},
    {'subset': 'FD003', 'candidate': 'temporal_lstm', 'role': 'competitor', 'model_family': 'lstm', 'preprocessing_run_id': TEMPORAL_PREPROCESSING_RUN_IDS['FD003'], 'training_run_id': 'd0c0bf4c7e7c430ab449d904de65bec2'},
    {'subset': 'FD004', 'candidate': 'temporal_xgboost', 'role': 'champion', 'model_family': 'tabular', 'preprocessing_run_id': TEMPORAL_PREPROCESSING_RUN_IDS['FD004'], 'training_run_id': '1b90ed34b6964e58bcef186c090cfc52'},
    {'subset': 'FD004', 'candidate': 'temporal_random_forest', 'role': 'competitor', 'model_family': 'tabular', 'preprocessing_run_id': TEMPORAL_PREPROCESSING_RUN_IDS['FD004'], 'training_run_id': '9765b1d0d6e34ec49360fa8b9bbeb09f'},
)

In [10]:
robustness_results = []

for candidate in ROBUSTNESS_CANDIDATES:
    validation_path = (PROCESSED_DATA_DIR / candidate['subset'] / candidate['preprocessing_run_id'] / 'validation')
    validation_dataframe = pd.read_parquet(validation_path)
    
    feature_columns = load_training_feature_columns(candidate['training_run_id'])
    validation_dataframe[feature_columns] = (validation_dataframe[feature_columns].fillna(0).astype(np.float32))
    
    model = load_training_model(candidate['training_run_id'])

    for seed in ROBUSTNESS_SEEDS:
        if candidate['model_family'] == 'lstm':
            metadata = select_pseudo_test_endpoints(validation_dataframe, seed=seed)
            sequences = build_endpoint_sequences(validation_dataframe, metadata, feature_columns, sequence_length=30)
            
            predictions = np.asarray([np.asarray(model.predict(sequence[np.newaxis, ...])).reshape(-1)[0] for sequence in sequences])
            targets = metadata['RUL']
            
        else:
            features, targets, metadata = prepare_pseudo_test_validation(validation_dataframe, feature_columns, seed=seed)
            predictions = model.predict(features)

        metrics = evaluate_predictions(targets, predictions)
        _, tail_metrics = evaluate_prediction_diagnostics(metadata, predictions)
        robustness_results.append({**candidate, 'seed': seed, **metrics, **tail_metrics})

robustness_by_seed = pd.DataFrame(robustness_results)
robustness_summary = (robustness_by_seed.groupby(['subset', 'candidate', 'role', 'preprocessing_run_id', 'training_run_id'], as_index=False)
                      .agg(mean_mae=('mae', 'mean'), 
                      median_mae=('mae', 'median'), 
                      mean_rmse=('rmse', 'mean'), 
                      median_rmse=('rmse', 'median'), 
                      mean_nasa_score=('nasa_score', 'mean'), 
                      median_nasa_score=('nasa_score', 'median'), 
                      worst_seed_nasa_score=('nasa_score', 'max'), 
                      worst_positive_error=('worst_positive_error', 'max')))

champion_scores = robustness_by_seed.loc[robustness_by_seed['role'] == 'champion', ['subset', 'seed', 'nasa_score']
                                         ].rename(columns={'nasa_score': 'champion_nasa_score'})

competitor_scores = robustness_by_seed.loc[robustness_by_seed['role'] == 'competitor', ['subset', 'seed', 'nasa_score']
                                           ].rename(columns={'nasa_score': 'competitor_nasa_score'})

win_rates = champion_scores.merge(competitor_scores, on=['subset', 'seed'])
win_rates = (win_rates.assign(champion_win=lambda frame: frame['champion_nasa_score'] < frame['competitor_nasa_score'])
             .groupby('subset', as_index=False)['champion_win']
             .mean()
             .rename(columns={'champion_win': 'champion_nasa_win_rate'}))

robustness_summary.merge(win_rates, on='subset', how='left').round(3)

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.3s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.7s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.2s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_job

,subset,candidate,role,preprocessing_run_id,training_run_id,mean_mae,median_mae,mean_rmse,median_rmse,mean_nasa_score,median_nasa_score,worst_seed_nasa_score,worst_positive_error,champion_nasa_win_rate
0,FD001,temporal_lstm,champion,20db40e34e4b42f392e931eab4ae1b7d,9d42f953ae0540a2b65d053137fd6361,11.257,10.367,16.967,17.441,155.897,173.238,239.123,49.728,0.3
1,FD001,temporal_xgboost,competitor,20db40e34e4b42f392e931eab4ae1b7d,d37624ea98f5459db673a69c5dc56951,11.213,11.463,15.797,16.650,109.719,102.083,192.131,43.889,0.3
2,FD002,temporal_lstm,champion,324277ebf695472886eaf129724e9833,490407b61beb40ffa97c0572bd04fc0c,12.234,11.939,16.837,16.624,366.415,355.712,571.830,51.757,0.9
3,FD002,temporal_xgboost,competitor,324277ebf695472886eaf129724e9833,1b7584f00aad4746a59ff0103f2448c3,14.426,14.232,19.733,19.981,667.395,666.531,1005.070,56.423,0.9
4,FD003,base_lstm,champion,61dd784103e24f408aea4486cb982a35,8f7a7a1c48ef4e11874b8555b6c7f4c7,10.975,10.803,15.818,15.393,130.956,72.491,443.369,53.596,0.7
5,FD003,temporal_lstm,competitor,246b7b1ac64d455298f08db0b0abd11d,d0c0bf4c7e7c430ab449d904de65bec2,12.248,12.380,17.604,18.239,211.120,176.033,573.923,59.067,0.7
6,FD004,temporal_random_forest,competitor,0bd8c9d6df084181b95c5048faa9ad5f,9765b1d0d6e34ec49360fa8b9bbeb09f,15.602,15.171,21.243,20.898,907.170,832.552,1758.322,70.480,0.7
7,FD004,temporal_xgboost,champion,0bd8c9d6df084181b95c5048faa9ad5f,1b90ed34b6964e58bcef186c090cfc52,15.513,15.188,21.063,20.516,762.804,673.860,1716.787,67.194,0.7
